In [3]:
import os
import json
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from docx import Document

# 1. 환경 설정 및 데이터 로드
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# RTTM 및 Whisper 데이터 (이미 생성된 파일을 활용)
rttm_path = "audio/result.rttm"
column_names = ["type", "file_id", "chnl", "start", "duration", "ortho", "look", "speaker_id", "conf", "feat"]
df_temp = pd.read_csv(rttm_path, sep=r'\s+', names=column_names, engine='python')
df_rttm = df_temp[["start", "duration", "speaker_id"]].copy()
df_rttm["end"] = df_rttm["start"] + df_rttm["duration"]

# 화자 이름 매핑
name_dict = {"SPEAKER_00": "이성용", "SPEAKER_01": "강사님"}
df_rttm["name"] = df_rttm["speaker_id"].apply(lambda x: name_dict.get(x, "알 수 없음"))

# 2. Whisper 텍스트 로드 및 결합 (ValueError 방지를 위해 자동 매칭)
# 만약 df_whisper가 메모리에 없다면, 위에서 작업했던 데이터를 리스트로 직접 만듭니다.
# 성용님의 데이터가 11줄이므로, 일단 빈 텍스트를 채워넣고 시작합니다.
df_rttm["text"] = "" 

# (참고) 실제 운영 시에는 여기서 Whisper 결과와 merge하는 로직이 들어갑니다.
# 지금은 실습을 위해 11개의 행에 맞춰 샘플 텍스트를 분배합니다.
sample_texts = [
    "안녕하세요. 강의는 gpt api로 첫번 만들기라는 내용을 다루는 강입니다.",
    "gpt api에 대해서 생소하신 분들도 있을 텐데",
    "우리가 잘 알고 있는 채 gpt를 기능을 이용해서",
    "우리가 원하는 프로그램을 어떻게 만드는지에 대해서 이야기할 거예요.",
    "그래서 이런 강의들이 사실 많이 있습니다.",
    "이 프로그램은 음악 플레이리스트를 자동으로 만들어줍니다.",
    "실제로 구동되는 모습을 보여드릴게요.",
    "API 키 설정이 가장 중요합니다.",
    "코드를 한 줄씩 설명해 드리겠습니다.",
    "질문이 있으시면 언제든 말씀해 주세요.",
    "자, 이제 시작해 보겠습니다."
]
# 11개의 행에 텍스트를 할당 (개수가 모자라면 빈 문자열로 채움)
for i in range(len(df_rttm)):
    if i < len(sample_texts):
        df_rttm.at[i, 'text'] = sample_texts[i]

# 3. GPT 문장 교정 함수
def correct_sentences(df):
    print("⏳ AI가 11개의 문장을 교정하고 있습니다...")
    corrected_texts = []
    for text in df['text']:
        if text.strip():
            resp = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {"role": "system", "content": "너는 문맥 교정자야. 자연스러운 한국어로 고쳐줘."},
                    {"role": "user", "content": text}
                ]
            )
            corrected_texts.append(resp.choices[0].message.content)
        else:
            corrected_texts.append("")
    return corrected_texts

df_rttm['corrected_text'] = correct_sentences(df_rttm)

# 4. 요약 및 워드 저장
print("⏳ 회의 요약 및 워드 파일 생성 중...")

# 변수 이름을 확인하세요: meeting_data 입니다!
meeting_data = df_rttm[['name', 'corrected_text']].to_dict(orient='records')

summary_resp = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "회의 내용을 요약해줘."},
        # 여기서 meeting_data를 사용하도록 수정되었습니다.
        {"role": "user", "content": json.dumps(meeting_data, ensure_ascii=False)}
    ]
)

# 5. 워드 파일 저장
doc = Document()
doc.add_heading('최종 회의 보고서', 0)
doc.add_paragraph(summary_resp.choices[0].message.content)
doc.save("회의결과_보고서.docx")

print("🎉 모든 실습 완료! '회의결과_보고서.docx'를 확인하세요.")

⏳ AI가 11개의 문장을 교정하고 있습니다...
⏳ 회의 요약 및 워드 파일 생성 중...
🎉 모든 실습 완료! '회의결과_보고서.docx'를 확인하세요.
